In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch
import pandas as pd
import numpy as np
os.environ["HUGGINGFACE_TOKEN"] = "INSERT TOKEN"
cache_dir='  '
#model_name = "microsoft/phi-4"
model_name="google/gemma-2-27b-it"
#model_name="meta-llama/Llama-3.2-3B-Instruct"
#model_name="meta-llama/Llama-3.1-8B-Instruct"
#model_name="CohereLabs/aya-expanse-32b"
#model_name="tiiuae/falcon-7b-instruct"
#model_name="mistralai/Mixtral-8x7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto", use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)

#model_name = "tiiuae/Falcon3-Mamba-7B-Instruct"

#model = AutoModelForCausalLM.from_pretrained(
#    model_name,
#    torch_dtype=torch.bfloat16,
#    device_map="auto",
# cache_dir=cache_dir
#)
#tokenizer = AutoTokenizer.from_pretrained(model_name)


/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

In [2]:
import pandas as pd
import numpy as np
file_path="clearv2.csv"
df=pd.read_csv(file_path)
df=df.drop(["Unnamed: 0"],axis=1)
#df=df.iloc[:,:40]
df.head()
df.columns

Index(['ID', 'Last Changed', 'Author', 'Title', 'Anthology', 'URL', 'Source',
       'Pub Year', 'Category', 'Location', 'License', 'MPAA\nMax',
       'MPAA \n#Max', 'MPAA\n#Avg', 'Excerpt', 'Google\nWC', 'Joon\nWC v1',
       'British WC', 'British Words', 'Sentence\nCount v1',
       'Sentence\nCount v2', 'Paragraphs', 'BT Easiness', 'BT s.e.',
       'Flesch-Reading-Ease', 'Flesch-Kincaid-Grade-Level',
       'Automated Readability Index', 'SMOG Readability',
       'New Dale-Chall Readability Formula', 'CAREC', 'CAREC_M', 'CARES',
       'CML2RI', 'firstPlace_pred', 'secondPlace_pred', 'thirdPlace_pred',
       'fourthPlace_pred', 'fifthPlace_pred', 'sixthPlace_pred',
       'Kaggle split', 'LLAMA_70B_vanilla', 'LLAMA_70B_avg',
       'LLAMA_70B_entropy', 'LLAMA_8B_vanilla', 'LLAMA_8B_avg',
       'LLAMA_8B_entropy', 'AYA_32B_vanilla', 'AYA_32B_avg', 'AYA_32B_entropy',
       'AYA_8B_vanilla', 'AYA_8B_avg', 'AYA_8B_entropy', 'LLAMA_3B_vanilla',
       'LLAMA_3B_avg', 'LLAMA_3B_ent

In [ ]:
### THIS FILE DOES GRADE LEVEL SCORING

In [ ]:
## LLAMA 
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,14]
    score=df.iloc[i,22]
    prompt=f"Rate the readability of the text between 3 (very easy to understand) and 12 (very challenging to understand) using the following scale: Considering the factors such as: sentence structure, vocabulary and grammar complexity, and overall clarity, which grade level is the text the most suitable for? For example, the answer would be 7 if the text is most suitable for students in seventh grade or older. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Only answer in this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['AYA_32B_vanilla']=scores_llm_vanilla
df['AYA_32B_avg']=scores_llm_avg
df['AYA_32B_entropy']=entropies
#print(corrs)
df.to_csv("clearv2.csv")

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


In [3]:
## mixtral, aya, gemma
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,14]
    score=df.iloc[i,22]
    prompt=f"Rate the readability of the text between 3 (very easy to understand) and 12 (very challenging to understand) using the following scale: Considering the factors such as: sentence structure, vocabulary and grammar complexity, and overall clarity, which grade level is the text the most suitable for? For example, the answer would be 7 if the text is most suitable for students in seventh grade or older. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Only answer in this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=5,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-2][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        if vanilla==1:
            logits2=scores[-1][0]
            probs2 = torch.softmax(logits2, dim=-1)
            top6_probs2, top6_ids2 = torch.topk(probs2, 6)
            top6_tokens2=tokenizer.convert_ids_to_tokens(top6_ids2.tolist())
            vanilla=int(top6_tokens2[0])+10
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        if vanilla<10:
            for (n,p) in zip(top6_tokens,top6_probs.tolist()):
                try:
                    temp=int(n)
                    if temp==1:
                        temp=10
                    avg+=int(temp)*p
                except ValueError:
                    #print(avg)
                    break
        if vanilla>=10:
            print(top6_tokens,top6_tokens2,top6_probs.tolist(),top6_probs2.tolist())
            for (n,p) in zip(top6_tokens,top6_probs.tolist()):
                    try:
                        temp=int(n)
                        if temp==1:
                            temp_p=p
                            temp=0
                        avg+=int(temp)*p
                    except ValueError:
                        #print(avg)
                        break
            for (n,p) in zip(top6_tokens2,top6_probs2.tolist()):
                    try:
                        temp=int(n)+10
                        avg+=int(temp)*p*temp_p
                    except ValueError:
                        #print(avg)
                        break    
            print(vanilla,avg)
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['gemma_27B_vanilla']=scores_llm_vanilla
df['gemma_27B_avg']=scores_llm_avg
df['gemma_27B_entropy']=entropies
#print(corrs)
df.to_csv("clearv2.csv")

The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.


['1', '9', '8', '7', '\n', '\n\n'] ['0', '1', '2', '\n\n', '▁', '\n'] [0.556779146194458, 0.43362003564834595, 0.008999496698379517, 0.0005404617986641824, 2.230752033938188e-05, 1.271044402528787e-05] [0.9705104231834412, 0.02930687554180622, 0.00017426504928153008, 1.507688239144045e-06, 7.121814746824384e-07, 6.690325449199008e-07]
10 9.562615529245726
['1', '9', '8', '7', '\n', '\n\n'] ['0', '1', '2', '\n\n', '3', '▁'] [0.9522475600242615, 0.04740961268544197, 0.00026482794783078134, 2.9712873583775945e-05, 1.918407178891357e-05, 9.06191417016089e-06] [0.8164291381835938, 0.18216997385025024, 0.0013908849796280265, 1.6285588344544522e-06, 1.1192910278623458e-06, 1.0514766017877264e-06]
10 10.127513321679018
['1', '9', '8', '7', '\n', '\n\n'] ['0', '1', '2', '\n\n', '▁', '\n'] [0.588106095790863, 0.4041990339756012, 0.007403163705021143, 0.0002533223305363208, 1.1130206075904425e-05, 7.186200946307508e-06] [0.9769430160522461, 0.02297549694776535, 7.312597881536931e-05, 1.2582019053

In [ ]:
# Gets confidence scores with grade level readability score. 

# Models: Phi or LLAMA 
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
verbal_confs=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,14]
    score=df.iloc[i,22]
    prompt=f"Rate the readability of the text between 3 (very easy to understand) and 12 (very challenging to understand) using the following scale: Considering the factors such as: sentence structure, vocabulary and grammar complexity, and overall clarity, which grade level is the text the most suitable for? For example, the answer would be 7 if the text is most suitable for students in seventh grade or older. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Only answer in this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=10,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits_last=scores[-3][0]
    logits_fifth_to_last = scores[-7][0]
    import torch
    probs_last = torch.softmax(logits_last, dim=-1)
    probs_fifth_to_last = torch.softmax(logits_fifth_to_last, dim=-1)
    entropy = -torch.sum(probs_fifth_to_last * torch.log(probs_fifth_to_last + 1e-12)).item()
    vocab_size = probs_fifth_to_last.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())

    top6_probs_fifth_to_last, top6_ids_fifth_to_last = torch.topk(probs_fifth_to_last, 6)
    top6_tokens_fifth_to_last=tokenizer.convert_ids_to_tokens(top6_ids_fifth_to_last.tolist())

    
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens_fifth_to_last[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM')
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
        
    avg2=0
    flag2=False
    try:
        vanilla_conf=int(top6_tokens_last[0])
    except ValueError:
        try:
            logits_last=scores[-2][0]
            probs_last = torch.softmax(logits_last, dim=-1)
            top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
            top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
            vanilla_conf=int(top6_tokens_last[0])
        except ValueError:
            try:
                logits_last=scores[-1][0]
                probs_last = torch.softmax(logits_last, dim=-1)
                top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                vanilla_conf=int(top6_tokens_last[0])
            except ValueError:
                
                print('ERROR - FIRST CONFIDENCE TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
                flag2=True
                verbal_confs.append('na')
    if flag2==False:
        for (n,p) in zip(top6_tokens_last,top6_probs_last.tolist()):
            try:
                avg2+=int(n)*p
            except ValueError:
                #print(avg)
                break
        verbal_confs.append(avg2)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
filtered_entropies, filtered_verbal_confs = zip(*[(e,v) for e,v in zip(entropies, verbal_confs) if e != 'na' and v != 'na'])
print(np.corrcoef(filtered_entropies, filtered_verbal_confs)[0, 1])

df['conf_llama_3b_vanilla']=scores_llm_vanilla
df['conf_llama_3b_avg']=scores_llm_avg
df['conf_llama_3b_entropy']=entropies
df['conf_llama_3b_verbal_conf']=verbal_confs
df.to_csv("clearv2.csv")

In [ ]:
##Confidence Gemma, Mixtral, AYA with grade level
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
verbal_confs=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,14]
    score=df.iloc[i,22]
    prompt=f"Rate the readability of the text between 3 (very easy to understand) and 12 (very challenging to understand) using the following scale: Considering the factors such as: sentence structure, vocabulary and grammar complexity, and overall clarity, which grade level is the text the most suitable for? For example, the answer would be 7 if the text is most suitable for students in seventh grade or older. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Only answer in this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=11,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits_last=scores[-4][0]
    logits_fifth_to_last = scores[-8][0]
    import torch
    probs_last = torch.softmax(logits_last, dim=-1)
    probs_fifth_to_last = torch.softmax(logits_fifth_to_last, dim=-1)
    entropy = -torch.sum(probs_fifth_to_last * torch.log(probs_fifth_to_last + 1e-12)).item()
    vocab_size = probs_fifth_to_last.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())

    top6_probs_fifth_to_last, top6_ids_fifth_to_last = torch.topk(probs_fifth_to_last, 6)
    top6_tokens_fifth_to_last=tokenizer.convert_ids_to_tokens(top6_ids_fifth_to_last.tolist())

    
    #avg=0
    #flag=False
    #try:
    #    vanilla=int(top6_tokens_fifth_to_last[0])
    #    scores_llm_vanilla.append(vanilla)
    #except ValueError:
    #    print('ERROR - FIRST TOKEN NOT NUM')
    #    scores_llm_vanilla.append('na')
    #    scores_llm_avg.append('na')
    #    flag=True
    #if flag==False:
    #    for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
    #        try:
    #            avg+=int(n)*p
    #        except ValueError:
    #            #print(avg)
    #            break
    #    scores_llm_avg.append(avg)











    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens_fifth_to_last[0])
        if vanilla==1:
            logits2=scores[-7][0]
            probs2 = torch.softmax(logits2, dim=-1)
            top6_probs2, top6_ids2 = torch.topk(probs2, 6)
            top6_tokens2=tokenizer.convert_ids_to_tokens(top6_ids2.tolist())
            vanilla=int(top6_tokens2[0])+10
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        if vanilla<10:
            for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
                try:
                    temp=int(n)
                    if temp==1:
                        temp=10
                    avg+=int(temp)*p
                except ValueError:
                    #print(avg)
                    break
        if vanilla>=10:
            #print(top6_tokens_fifth_to_last,top6_tokens2,top6_probs_fifth_to_last.tolist(),top6_probs2.tolist())
            for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
                    try:
                        temp=int(n)
                        if temp==1:
                            temp_p=p
                            temp=0
                        avg+=int(temp)*p
                    except ValueError:
                        #print(avg)
                        break
            for (n,p) in zip(top6_tokens2,top6_probs2.tolist()):
                    try:
                        temp=int(n)+10
                        avg+=int(temp)*p*temp_p
                    except ValueError:
                        #print(avg)
                        break    
            #print(vanilla,avg)
        scores_llm_avg.append(avg)






        


    avg2=0
    flag2=False
    try:
        vanilla_conf=int(top6_tokens_last[0])
    except ValueError:
        try:
            logits_last=scores[-3][0]
            probs_last = torch.softmax(logits_last, dim=-1)
            top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
            top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
            vanilla_conf=int(top6_tokens_last[0])
        except ValueError:
            try:
                logits_last=scores[-2][0]
                probs_last = torch.softmax(logits_last, dim=-1)
                top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                vanilla_conf=int(top6_tokens_last[0])
            except ValueError:
                try:
                    logits_last=scores[-1][0]
                    probs_last = torch.softmax(logits_last, dim=-1)
                    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                    vanilla_conf=int(top6_tokens_last[0])
                except ValueError:
                    
                    print('ERROR - FIRST CONFIDENCE TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
                    flag2=True
                    verbal_confs.append('na')
    if flag2==False:
        for (n,p) in zip(top6_tokens_last,top6_probs_last.tolist()):
            try:
                avg2+=int(n)*p
            except ValueError:
                #print(avg)
                break
        verbal_confs.append(avg2)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
filtered_entropies, filtered_verbal_confs = zip(*[(e,v) for e,v in zip(entropies, verbal_confs) if e != 'na' and v != 'na'])
print(np.corrcoef(filtered_entropies, filtered_verbal_confs)[0, 1])

df['conf_aya_32B_vanilla']=scores_llm_vanilla
df['conf_aya_32B_avg']=scores_llm_avg
df['conf_aya_32B_entropy']=entropies
df['conf_aya_32B_verbal_conf']=verbal_confs
df.to_csv("clearv2.csv")